In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import numpy as np
from sklearn.metrics import classification_report
import time

# 1. Fix the warnings and optimize device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Updated hyperparameters for faster training
BATCH_SIZE = 64  # Increased batch size
LEARNING_RATE = 0.001
EPOCHS = 15
IMAGE_SIZE = 224

# 3. Dataset paths - verify this matches your actual path structure
data_dir = '/kaggle/input/sugarcane-leaf-disease-dataset'
classes = ['Healthy', 'Mosaic', 'RedRot', 'Rust', 'Yellow']

# 4. More efficient transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 5. Optimized Dataset class
class SugarcaneDataset(Dataset):
    def __init__(self, root_dir, transform=None, split='train', split_ratio=0.8):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = classes
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.samples = []
        
        # Load image paths with error handling
        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)
            try:
                img_names = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                self.samples.extend([(os.path.join(class_path, img_name), self.class_to_idx[class_name]) 
                                for img_name in img_names])
            except FileNotFoundError:
                print(f"Warning: Class directory {class_path} not found")
                continue
        
        # Shuffle and split
        np.random.seed(42)
        np.random.shuffle(self.samples)
        split_idx = int(len(self.samples) * split_ratio)
        self.samples = self.samples[:split_idx] if split == 'train' else self.samples[split_idx:]
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading image {img_path}: {str(e)}")
            # Return a random image if there's an error
            return self.transform(Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE))), 0

# 6. Create datasets with progress indication
print("Loading datasets...")
start_time = time.time()
train_dataset = SugarcaneDataset(data_dir, transform=data_transforms['train'], split='train')
val_dataset = SugarcaneDataset(data_dir, transform=data_transforms['val'], split='val')
print(f"Datasets loaded in {time.time()-start_time:.2f} seconds")
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

# 7. Optimized DataLoader with pin_memory for GPU
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                         num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=4, pin_memory=True)

# 8. Updated model loading with current torchvision syntax
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))
model = model.to(device)

# 9. Training setup with gradient clipping
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=3, factor=0.1)

# 10. Improved training function with progress tracking
def train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS):
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0
        processed_samples = 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # Gradient clipping
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            processed_samples += inputs.size(0)
            
            if batch_idx % 10 == 0:
                print(f'Batch {batch_idx}/{len(train_loader)} - '
                      f'Loss: {loss.item():.4f}', end='\r')
        
        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = running_corrects.double() / len(train_dataset)
        print(f'\nTrain Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_epoch_loss = val_loss / len(val_dataset)
        val_epoch_acc = val_corrects.double() / len(val_dataset)
        
        scheduler.step(val_epoch_acc)  # Update learning rate
        
        print(f'Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}')
        
        # Save best model
        if val_epoch_acc > best_acc:
            best_acc = val_epoch_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print("New best model saved!")
    
    print(f'\nBest val Acc: {best_acc:.4f}')
    
    # Classification report
    print('\nClassification Report:')
    print(classification_report(all_labels, all_preds, target_names=classes))
    
    return model

# Train the model
print("\nStarting training...")
start_time = time.time()
model = train_model(model, criterion, optimizer, scheduler, EPOCHS)
print(f"\nTraining completed in {(time.time()-start_time)/60:.2f} minutes")

# Save the final model
torch.save(model.state_dict(), 'final_model.pth')
print("Final model saved as 'final_model.pth'")

Using device: cpu
Loading datasets...
Datasets loaded in 0.07 seconds
Training samples: 2016, Validation samples: 505


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]



Starting training...

Epoch 1/15
----------
Batch 30/32 - Loss: 0.7427
Train Loss: 0.7549 Acc: 0.7361
Val Loss: 1.4651 Acc: 0.6970
New best model saved!

Epoch 2/15
----------
Batch 30/32 - Loss: 0.2963
Train Loss: 0.3934 Acc: 0.8666
Val Loss: 0.6412 Acc: 0.7941
New best model saved!

Epoch 3/15
----------
Batch 30/32 - Loss: 0.3659
Train Loss: 0.3430 Acc: 0.8938
Val Loss: 0.3323 Acc: 0.9208
New best model saved!

Epoch 4/15
----------
Batch 30/32 - Loss: 0.1253
Train Loss: 0.3111 Acc: 0.8978
Val Loss: 0.2810 Acc: 0.9010

Epoch 5/15
----------
Batch 30/32 - Loss: 0.1601
Train Loss: 0.2495 Acc: 0.9206
Val Loss: 0.5719 Acc: 0.8317

Epoch 6/15
----------
Batch 30/32 - Loss: 0.1322
Train Loss: 0.2172 Acc: 0.9276
Val Loss: 0.3198 Acc: 0.9089

Epoch 7/15
----------
Batch 30/32 - Loss: 0.3207
Train Loss: 0.2274 Acc: 0.9291
Val Loss: 0.3642 Acc: 0.8911

Epoch 8/15
----------
Batch 30/32 - Loss: 0.0566
Train Loss: 0.1338 Acc: 0.9544
Val Loss: 0.1408 Acc: 0.9545
New best model saved!

Epoch 9/1